<a href="https://colab.research.google.com/github/san9roy/Sector-Coupled-Urban-Energy-System-Simulation/blob/main/Sector_Coupled_Urban_Energy_System_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
"""
Sector-Coupled Urban Energy System Simulation (Berlin-style microgrid)
=====================================================================

What this code provides:
- Synthetic hourly profiles for a full year (load, PV, outdoor temp, heat demand, TOU price, CO2 intensity)
- Physics-based component models:
    * Battery (SOC dynamics, power limits, efficiency, SOC bounds)
    * Heat pump (COP as a function of outdoor temperature)
    * Thermal tank (energy state, min/max bounds, standby losses)
- Dispatch strategies:
    * Rule-based controller (PV self-consumption + optional thermal precharge)
    * Optimization-based dispatch (daily rolling-horizon Linear Program via SciPy)
- Outputs:
    * Hourly time series of all flows, states, costs, emissions + conservation residuals
    * CSV exports for baseline / rule / optimization
- Plots:
    * Weekly power flow plots (winter and summer)
    * Electricity balance residual plot (validation)
    * SOC & tank energy time series (winter and summer)
    * SOC histogram
    * Cumulative cost and cumulative CO2 (rule vs opt if available)
    * Cost by hour-of-day
    * CO2 by month
    * Grid import duration curve
    * Grid import by hour-of-day
    * Dispatch overlay (rule vs opt)
    * Sensitivity curves (battery size vs cost/CO2/peak)
    * Scenario comparison (cost/CO2/peak)

Notes:
- The synthetic profiles are "Berlin-shaped" but not real data.
- The LP uses a simplified linear representation of tank loss (alpha weighting) and exogenous COP.
- Matplotlib styles/colors are left as defaults (no seaborn, no manual color settings).

Run:
  python main.py

Artifacts:
  results/
    results_baseline_hourly.csv
    results_rule_hourly.csv
    results_opt_hourly.csv (if SciPy available)
    sensitivity_table.csv
    scenario_summary.csv
  results/figures/
    (png files)

"""

'\nSector-Coupled Urban Energy System Simulation (Berlin-style microgrid)\n=====================================================================\n\nWhat this code provides:\n- Synthetic hourly profiles for a full year (load, PV, outdoor temp, heat demand, TOU price, CO2 intensity)\n- Physics-based component models:\n    * Battery (SOC dynamics, power limits, efficiency, SOC bounds)\n    * Heat pump (COP as a function of outdoor temperature)\n    * Thermal tank (energy state, min/max bounds, standby losses)\n- Dispatch strategies:\n    * Rule-based controller (PV self-consumption + optional thermal precharge)\n    * Optimization-based dispatch (daily rolling-horizon Linear Program via SciPy)\n- Outputs:\n    * Hourly time series of all flows, states, costs, emissions + conservation residuals\n    * CSV exports for baseline / rule / optimization\n- Plots:\n    * Weekly power flow plots (winter and summer)\n    * Electricity balance residual plot (validation)\n    * SOC & tank energy time

In [6]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Tuple, Literal, Optional, List
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# SciPy is optional: if not installed, optimization plots won't be generated
try:
    from scipy.optimize import linprog
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

In [7]:
# =========================================================
# PARAMETERS /  DEFINITIONS
# =========================================================

@dataclass(frozen=True)
class SystemParams:
    """
    Global parameter container for all components and signals.

    Keeping parameters in one dataclass makes the project:
    - Reproducible (easy to save/serialize)
    - Easy to sweep (sensitivity analysis)
    - Clean (no scattered constants)
    """
    # Simulation time step
    dt_h: float = 1.0

    # Battery parameters
    batt_capacity_kwh: float = 10.0
    batt_p_ch_max_kw: float = 3.0
    batt_p_dis_max_kw: float = 3.0
    batt_eta_ch: float = 0.95
    batt_eta_dis: float = 0.95
    batt_soc_min: float = 0.10
    batt_soc_max: float = 0.90

    # Heat pump parameters
    hp_p_max_kw: float = 4.0
    hp_cop_min: float = 1.8
    hp_cop_max: float = 4.0

    # Thermal tank parameters
    tank_capacity_kwh_th: float = 30.0
    tank_min_kwh_th: float = 3.0
    tank_loss_per_h: float = 0.01

    # Grid economics
    feed_in_tariff_eur_per_kwh: float = 0.08


@dataclass
class SystemState:
    """
    Dynamic states of the system.

    SOC is kept in [0..1] when battery exists.
    If battery is disabled (capacity ~0), we keep SOC=0.0 and ignore charge/discharge actions.
    """
    soc: float
    tank_kwh_th: float


In [8]:
# =========================================================
# SIGNALS / PROFILES (SYNTHETIC)
# =========================================================

def generate_synthetic_berlin_profiles(year: int = 2025, seed: int = 42) -> pd.DataFrame:
    """
    Generate synthetic hourly profiles for an entire year.

    Output columns:
      - load_base_kw: base electricity demand (kW)
      - pv_kw: PV production (kW)
      - t_out_C: outdoor temperature (°C)
      - heat_demand_kwh_th: hourly space heating energy demand (kWh_th)
      - price_import_eur_per_kwh: TOU electricity import price (€/kWh)
      - co2_g_per_kwh: grid carbon intensity (gCO2/kWh)

    Design logic:
      - Electricity load: morning + evening peaks, weekend uplift, noise
      - PV: diurnal sun curve times seasonal factor, noise
      - Temperature: seasonal sinusoid + diurnal + noise
      - Heat demand: UA*(T_in - T_out)+, interpreted as kWh_th/h (because dt=1h)
      - Price: TOU with evening peak, night discount, mild seasonality
      - CO2: seasonal + evening peak concentration, noise
    """
    rng = np.random.default_rng(seed)

    # 'h' is the recommended hourly frequency (uppercase 'H' is deprecated in pandas)
    idx = pd.date_range(f"{year}-01-01", f"{year+1}-01-01", freq="h", inclusive="left")
    n = len(idx)

    hour = idx.hour.values
    dayofyear = idx.dayofyear.values
    weekday = idx.weekday.values

    # Outdoor temperature model (Berlin-ish, not real data)
    seasonal = 10 + 10 * np.sin(2 * np.pi * (dayofyear - 80) / 365)
    diurnal = 2 * np.sin(2 * np.pi * (hour - 14) / 24)
    t_out = seasonal + diurnal + rng.normal(0, 1.2, n)

    # Base electricity load (kW)
    morning_peak = 0.6 * np.exp(-0.5 * ((hour - 8) / 2.0) ** 2)
    evening_peak = 1.0 * np.exp(-0.5 * ((hour - 19) / 2.5) ** 2)
    base = 0.6 + morning_peak + evening_peak
    weekend = np.where(weekday >= 5, 1.10, 1.0)
    load_base = (base * weekend) + rng.normal(0, 0.05, n)
    load_base = np.clip(load_base, 0.2, None)

    # PV model (kW)
    sun = np.clip(np.sin(np.pi * (hour - 6) / 12), 0, None)
    pv_season = np.clip(0.15 + 0.85 * np.sin(2 * np.pi * (dayofyear - 80) / 365), 0, None)
    pv = 3.5 * sun * pv_season + rng.normal(0, 0.05, n)
    pv = np.clip(pv, 0.0, None)

    # Heat demand model (kWh_th per hour)
    T_in = 20.0
    UA = 0.45
    heat_kw_th = UA * np.clip(T_in - t_out, 0, None)
    heat_kwh_th = heat_kw_th  # because dt=1 hour

    # Time-of-use price (€/kWh)
    base_price = 0.26 + 0.03 * np.sin(2 * np.pi * (dayofyear - 15) / 365)
    peak_add = 0.10 * ((hour >= 17) & (hour <= 21)).astype(float)
    night_discount = -0.05 * ((hour >= 0) & (hour <= 5)).astype(float)
    price_import = np.clip(base_price + peak_add + night_discount, 0.12, None)

    # CO2 intensity (g/kWh)
    co2_base = 320 + 80 * np.sin(2 * np.pi * (dayofyear + 20) / 365)
    co2_peak = 90 * ((hour >= 17) & (hour <= 21)).astype(float)
    co2 = np.clip(co2_base + co2_peak + rng.normal(0, 10, n), 150, 600)

    return pd.DataFrame(
        {
            "load_base_kw": load_base,
            "pv_kw": pv,
            "t_out_C": t_out,
            "heat_demand_kwh_th": heat_kwh_th,
            "price_import_eur_per_kwh": price_import,
            "co2_g_per_kwh": co2,
        },
        index=idx,
    )


In [9]:
# =========================================================
# COMPONENT MODELS (PHYSICS + CONSTRAINTS)
# =========================================================

def cop_from_temp(t_out_C: float, p: SystemParams) -> float:
    """
    Heat pump COP model: clipped linear relationship vs outdoor temperature.
    - Warmer outside -> higher COP
    - Clipped to [hp_cop_min, hp_cop_max]
    """
    cop = 3.2 - 0.06 * (10 - t_out_C)
    return float(np.clip(cop, p.hp_cop_min, p.hp_cop_max))


def battery_step(state: SystemState, p_ch_kw: float, p_dis_kw: float, p: SystemParams) -> SystemState:
    """
    Update battery SOC by applying charge/discharge power for one timestep.

    Battery dynamics (energy form):
      E_{t+1} = E_t + (eta_ch * P_ch - P_dis/eta_dis) * dt

    Constraints enforced:
      - P_ch, P_dis >= 0 (caller should not pass negatives; we clip anyway)
      - E_min <= E_{t+1} <= E_max
      - If battery is disabled (capacity ~0), return SOC=0 and ignore actions
    """
    if p.batt_capacity_kwh <= 1e-12 or (p.batt_p_ch_max_kw <= 1e-12 and p.batt_p_dis_max_kw <= 1e-12):
        return SystemState(soc=0.0, tank_kwh_th=state.tank_kwh_th)

    p_ch_kw = max(0.0, float(p_ch_kw))
    p_dis_kw = max(0.0, float(p_dis_kw))

    # Convert SOC -> energy
    e_kwh = state.soc * p.batt_capacity_kwh

    # Apply energy update
    e_kwh_next = e_kwh + (p.batt_eta_ch * p_ch_kw - (p_dis_kw / p.batt_eta_dis)) * p.dt_h

    # Enforce SOC bounds in energy space
    e_min = p.batt_soc_min * p.batt_capacity_kwh
    e_max = p.batt_soc_max * p.batt_capacity_kwh
    e_kwh_next = float(np.clip(e_kwh_next, e_min, e_max))

    return SystemState(soc=e_kwh_next / p.batt_capacity_kwh, tank_kwh_th=state.tank_kwh_th)


def thermal_tank_step(state: SystemState, heat_in_kwh_th: float, heat_out_kwh_th: float, p: SystemParams) -> SystemState:
    """
    Thermal tank energy update with standby losses.

    Tank dynamics:
      loss = loss_rate * E_t
      E_{t+1} = E_t + heat_in - heat_out - loss

    Constraints:
      tank_min_kwh_th <= E_{t+1} <= tank_capacity_kwh_th
    """
    heat_in_kwh_th = max(0.0, float(heat_in_kwh_th))
    heat_out_kwh_th = max(0.0, float(heat_out_kwh_th))

    loss = p.tank_loss_per_h * state.tank_kwh_th
    e_next = state.tank_kwh_th + heat_in_kwh_th - heat_out_kwh_th - loss

    e_next = float(np.clip(e_next, p.tank_min_kwh_th, p.tank_capacity_kwh_th))
    return SystemState(soc=state.soc, tank_kwh_th=e_next)


In [10]:
# =========================================================
#  DISPATCH: RULE-BASED CONTROL
# =========================================================

def dispatch_rule_based(
    pv_kw: float,
    load_kw: float,
    heat_demand_kwh_th: float,
    t_out_C: float,
    price_import: float,
    co2_g_per_kwh: float,
    state: SystemState,
    p: SystemParams,
    thermal_precharge: bool = True,
) -> Dict[str, float]:
    """
    Rule-based controller (heuristic dispatch).

    Thermal logic:
      1) Discharge tank above minimum to serve heat demand.
      2) Run heat pump to cover remaining heat demand.
      3) Optional: precharge tank (extra HP operation) when PV surplus exists or grid is cheap/clean.

    Electric logic:
      - PV -> load
      - Surplus PV -> battery charge (up to limit)
      - Remaining deficit -> battery discharge (up to limit)
      - Grid imports/exports close the balance (computed in simulation engine)

    Returns a dict of control actions for the current hour.
    """
    cop = cop_from_temp(t_out_C, p)

    # --- Serve heat demand from tank first
    tank_available = max(0.0, state.tank_kwh_th - p.tank_min_kwh_th)
    tank_dis = min(heat_demand_kwh_th, tank_available)
    remaining_heat = heat_demand_kwh_th - tank_dis

    # --- Heat pump covers remaining heat
    hp_p_for_demand = remaining_heat / cop if cop > 0 else 0.0
    hp_p_kw = min(p.hp_p_max_kw, hp_p_for_demand)

    hp_heat_out = hp_p_kw * cop * p.dt_h

    # Any heat produced beyond remaining_heat is stored
    heat_in = max(0.0, hp_heat_out - remaining_heat)

    # --- Estimate PV surplus to decide about precharge
    total_electric_load_kw = load_kw + hp_p_kw
    pv_to_load = min(pv_kw, total_electric_load_kw)
    residual_load = total_electric_load_kw - pv_to_load
    pv_surplus = pv_kw - pv_to_load

    # --- Optional pre-charge to demonstrate thermal flexibility
    if thermal_precharge and p.tank_capacity_kwh_th > p.tank_min_kwh_th + 1e-6:
        tank_headroom = max(0.0, p.tank_capacity_kwh_th - state.tank_kwh_th)

        # Simple “cheap/clean” thresholds
        cheap = price_import <= 0.22
        clean = co2_g_per_kwh <= 280

        if pv_surplus > 0.2 or cheap or clean:
            # Extra HP power limited by remaining HP capacity and tank headroom
            extra_hp_p = min(
                p.hp_p_max_kw - hp_p_kw,
                (tank_headroom / (cop * p.dt_h)) if cop > 0 else 0.0
            )
            extra_hp_p = max(0.0, extra_hp_p)

            # If no PV surplus, only precharge when cheap/clean
            if pv_surplus <= 0.2 and not (cheap or clean):
                extra_hp_p = 0.0

            if extra_hp_p > 0:
                hp_p_kw += extra_hp_p
                heat_in += extra_hp_p * cop * p.dt_h

                # Recompute surplus with increased HP power
                total_electric_load_kw = load_kw + hp_p_kw
                pv_to_load = min(pv_kw, total_electric_load_kw)
                residual_load = total_electric_load_kw - pv_to_load
                pv_surplus = pv_kw - pv_to_load

    # --- Battery actions (requested powers; actual effect may clip due SOC)
    batt_ch_kw = min(p.batt_p_ch_max_kw, pv_surplus) if p.batt_p_ch_max_kw > 0 else 0.0
    batt_dis_kw = min(p.batt_p_dis_max_kw, residual_load) if p.batt_p_dis_max_kw > 0 else 0.0

    return {
        "batt_ch_kw": float(batt_ch_kw),
        "batt_dis_kw": float(batt_dis_kw),
        "hp_p_kw": float(hp_p_kw),
        "tank_dis_kwh_th": float(tank_dis),
        "tank_charge_kwh_th": float(heat_in),
        "cop": float(cop),
    }

In [11]:
# =========================================================
#  DISPATCH: OPTIMIZATION (DAILY ROLLING-HORIZON LP)
# =========================================================

def dispatch_optimized_day_lp(
    day_df: pd.DataFrame,
    init_state: SystemState,
    p: SystemParams,
    co2_penalty_eur_per_kg: float = 0.0,
) -> pd.DataFrame:
    """
    Daily linear program (24-hour horizon).

    Decision variables per hour:
      batt_ch, batt_dis, grid_imp, grid_exp, hp_p, tank_charge, tank_dis

    Constraints:
      - Electricity balance each hour
      - Thermal balance each hour
      - Battery energy bounds across horizon (cumulative)
      - Tank energy bounds across horizon (alpha loss approximation)

    Objective:
      min Σ(import*price - export*FIT) + Σ(co2_penalty * kgCO2(import))

    Returns:
      DataFrame of hourly actions for the day.
    """
    if not SCIPY_AVAILABLE:
        raise RuntimeError("SciPy not available; cannot run LP optimization.")

    H = len(day_df)
    if H != 24:
        raise ValueError(f"Expected 24 hours, got {H}.")

    dt = p.dt_h
    pv = day_df["pv_kw"].to_numpy()
    load = day_df["load_base_kw"].to_numpy()
    heat_d = day_df["heat_demand_kwh_th"].to_numpy()
    price = day_df["price_import_eur_per_kwh"].to_numpy()
    co2_g = day_df["co2_g_per_kwh"].to_numpy()

    # COP is exogenous here (still consistent with simulation)
    cop = np.array([cop_from_temp(t, p) for t in day_df["t_out_C"].to_numpy()])

    # Variable ordering:
    # x = [bch(0..H-1), bdis, gimp, gexp, hp, tchg, tdis]
    nvar = 7 * H

    def sl(i: int, block: int) -> int:
        """Index helper: variable position for hour i in a given block."""
        return block * H + i

    # -------------------------
    # Objective coefficients
    # -------------------------
    c = np.zeros(nvar)
    for i in range(H):
        # import cost
        c[sl(i, 2)] = price[i] * dt
        # export revenue (negative cost)
        c[sl(i, 3)] = -p.feed_in_tariff_eur_per_kwh * dt
        # optional CO2 penalty
        if co2_penalty_eur_per_kg > 0:
            kg_per_kw = (co2_g[i] / 1000.0) * dt
            c[sl(i, 2)] += co2_penalty_eur_per_kg * kg_per_kw

    # -------------------------
    # Variable bounds
    # -------------------------
    bounds = []
    for _ in range(H): bounds.append((0, max(0.0, p.batt_p_ch_max_kw)))   # batt_ch
    for _ in range(H): bounds.append((0, max(0.0, p.batt_p_dis_max_kw)))  # batt_dis
    for _ in range(H): bounds.append((0, None))                           # grid_import
    for _ in range(H): bounds.append((0, None))                           # grid_export
    for _ in range(H): bounds.append((0, max(0.0, p.hp_p_max_kw)))         # hp_p
    for _ in range(H): bounds.append((0, None))                           # tank_charge
    for _ in range(H): bounds.append((0, None))                           # tank_dis

    # -------------------------
    # Equality constraints
    # -------------------------
    A_eq, b_eq = [], []

    # Electricity balance:
    # pv + batt_dis + grid_imp = load + hp + batt_ch + grid_exp
    for i in range(H):
        row = np.zeros(nvar)
        row[sl(i, 1)] = 1.0   # batt_dis
        row[sl(i, 2)] = 1.0   # grid_imp
        row[sl(i, 4)] = -1.0  # hp
        row[sl(i, 0)] = -1.0  # batt_ch
        row[sl(i, 3)] = -1.0  # grid_exp
        A_eq.append(row)
        b_eq.append(load[i] - pv[i])

    # Thermal balance:
    # tank_dis + hp*COP*dt = heat_d + tank_charge
    for i in range(H):
        row = np.zeros(nvar)
        row[sl(i, 6)] = 1.0            # tank_dis
        row[sl(i, 4)] = cop[i] * dt    # hp -> heat
        row[sl(i, 5)] = -1.0           # tank_charge
        A_eq.append(row)
        b_eq.append(heat_d[i])

    A_eq = np.asarray(A_eq)
    b_eq = np.asarray(b_eq)

    # -------------------------
    # Inequality constraints (storage bounds across horizon)
    # -------------------------
    A_ub, b_ub = [], []

    # Battery cumulative energy bounds
    if p.batt_capacity_kwh > 1e-12 and not (p.batt_p_ch_max_kw <= 1e-12 and p.batt_p_dis_max_kw <= 1e-12):
        E0 = init_state.soc * p.batt_capacity_kwh
        Emin = p.batt_soc_min * p.batt_capacity_kwh
        Emax = p.batt_soc_max * p.batt_capacity_kwh

        for k in range(1, H + 1):
            # E_k = E0 + sum_{i<k}(eta_ch*bch_i - bdis_i/eta_dis)*dt
            row = np.zeros(nvar)
            for i in range(k):
                row[sl(i, 0)] += p.batt_eta_ch * dt
                row[sl(i, 1)] += -(dt / p.batt_eta_dis)

            # E_k <= Emax
            A_ub.append(row.copy())
            b_ub.append(Emax - E0)

            # E_k >= Emin  -> -row*x <= E0 - Emin
            A_ub.append((-row).copy())
            b_ub.append(E0 - Emin)

    # Tank energy bounds (approximate loss with alpha weights)
    T0 = init_state.tank_kwh_th
    Tmin = p.tank_min_kwh_th
    Tmax = p.tank_capacity_kwh_th
    alpha = (1.0 - p.tank_loss_per_h)

    for k in range(1, H + 1):
        # T_k = alpha^k*T0 + sum_{i<k} alpha^{k-1-i}*(tchg_i - tdis_i)
        row = np.zeros(nvar)
        for i in range(k):
            w = alpha ** (k - 1 - i)
            row[sl(i, 5)] += w      # tank_charge
            row[sl(i, 6)] += -w     # tank_dis

        Tk0 = (alpha ** k) * T0

        # T_k <= Tmax
        A_ub.append(row.copy())
        b_ub.append(Tmax - Tk0)

        # T_k >= Tmin
        A_ub.append((-row).copy())
        b_ub.append(Tk0 - Tmin)

    A_ub = np.asarray(A_ub) if len(A_ub) else None
    b_ub = np.asarray(b_ub) if len(b_ub) else None

    # Solve LP
    res = linprog(c=c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not res.success:
        raise RuntimeError(f"LP failed: {res.message}")

    x = res.x
    actions = pd.DataFrame(index=day_df.index)
    actions["batt_ch_kw"] = [x[sl(i, 0)] for i in range(H)]
    actions["batt_dis_kw"] = [x[sl(i, 1)] for i in range(H)]
    actions["grid_import_kw"] = [x[sl(i, 2)] for i in range(H)]
    actions["grid_export_kw"] = [x[sl(i, 3)] for i in range(H)]
    actions["hp_p_kw"] = [x[sl(i, 4)] for i in range(H)]
    actions["tank_charge_kwh_th"] = [x[sl(i, 5)] for i in range(H)]
    actions["tank_dis_kwh_th"] = [x[sl(i, 6)] for i in range(H)]
    actions["cop"] = cop
    actions["lp_objective_eur"] = res.fun
    return actions


In [12]:
# =========================================================
# SIMULATION ENGINE (ONE-YEAR HOURLY LOOP)
# =========================================================

def simulate(
    df: pd.DataFrame,
    p: SystemParams,
    strategy: Literal["rule", "lp"] = "rule",
    co2_penalty_eur_per_kg: float = 0.0,
    init_state: Optional[SystemState] = None,
    thermal_precharge_rule: bool = True,
) -> pd.DataFrame:
    """
    Main hourly simulation routine.

    - Reads exogenous profiles (pv, load, heat demand, price, co2, temperature)
    - Chooses dispatch actions (rule or LP)
    - Applies physics-based state updates for battery and thermal tank
    - Computes grid import/export, costs, emissions
    - Records time-series outputs

    Returns:
      DataFrame indexed by hour containing:
      - flows (grid, PV, battery, HP, tank)
      - states (SOC, tank energy)
      - costs and emissions
      - energy balance residual for validation
    """
    if init_state is None:
        init_state = SystemState(
            soc=0.5 if p.batt_capacity_kwh > 1e-12 else 0.0,
            tank_kwh_th=0.5 * p.tank_capacity_kwh_th if p.tank_capacity_kwh_th > p.tank_min_kwh_th else p.tank_min_kwh_th,
        )

    state = SystemState(soc=init_state.soc, tank_kwh_th=init_state.tank_kwh_th)

    # Precompute LP actions for each day if optimization is requested
    lp_actions: Optional[pd.DataFrame] = None
    if strategy == "lp":
        if not SCIPY_AVAILABLE:
            raise RuntimeError("SciPy not available; cannot run LP strategy.")

        all_actions = []

        # Rolling horizon daily: solve 24h LP, then move to next day with updated state
        for _, day_df in df.groupby(df.index.date):
            if len(day_df) != 24:
                continue

            actions = dispatch_optimized_day_lp(day_df, state, p, co2_penalty_eur_per_kg)
            all_actions.append(actions)

            # Advance state using the planned actions to set next day's initial state
            for t in actions.index:
                a = actions.loc[t]
                state = battery_step(state, a["batt_ch_kw"], a["batt_dis_kw"], p)
                state = thermal_tank_step(state, a["tank_charge_kwh_th"], a["tank_dis_kwh_th"], p)

        lp_actions = pd.concat(all_actions).sort_index()

        # Reset state for the actual simulation pass (so the recorded outputs are consistent)
        state = SystemState(soc=init_state.soc, tank_kwh_th=init_state.tank_kwh_th)

    rows = []
    for t, r in df.iterrows():
        pv_kw = float(r["pv_kw"])
        load_kw = float(r["load_base_kw"])
        heat_d = float(r["heat_demand_kwh_th"])
        t_out = float(r["t_out_C"])
        price = float(r["price_import_eur_per_kwh"])
        co2_g = float(r["co2_g_per_kwh"])

        # -------------------------
        # Choose dispatch actions
        # -------------------------
        if strategy == "rule":
            a = dispatch_rule_based(
                pv_kw, load_kw, heat_d, t_out, price, co2_g, state, p,
                thermal_precharge=thermal_precharge_rule
            )
            batt_ch = a["batt_ch_kw"]
            batt_dis = a["batt_dis_kw"]
            hp_p = a["hp_p_kw"]
            tank_dis = a["tank_dis_kwh_th"]
            tank_ch = a["tank_charge_kwh_th"]
            cop = a["cop"]

        else:
            # Optimization (fallback to rule-based if missing hour, e.g., incomplete day)
            assert lp_actions is not None
            if t not in lp_actions.index:
                a = dispatch_rule_based(
                    pv_kw, load_kw, heat_d, t_out, price, co2_g, state, p,
                    thermal_precharge=thermal_precharge_rule
                )
                batt_ch, batt_dis, hp_p = a["batt_ch_kw"], a["batt_dis_kw"], a["hp_p_kw"]
                tank_dis, tank_ch, cop = a["tank_dis_kwh_th"], a["tank_charge_kwh_th"], a["cop"]
            else:
                a = lp_actions.loc[t]
                batt_ch = float(a["batt_ch_kw"])
                batt_dis = float(a["batt_dis_kw"])
                hp_p = float(a["hp_p_kw"])
                tank_ch = float(a["tank_charge_kwh_th"])
                tank_dis = float(a["tank_dis_kwh_th"])
                cop = float(a["cop"])

        # -------------------------
        # Apply physics updates
        # -------------------------
        state_before = SystemState(state.soc, state.tank_kwh_th)

        # Battery update (SOC clipping may cause "difference" vs requested powers)
        state = battery_step(state, batt_ch, batt_dis, p)

        # Track battery energy delta (for analysis/validation)
        e_before = state_before.soc * p.batt_capacity_kwh if p.batt_capacity_kwh > 1e-12 else 0.0
        e_after = state.soc * p.batt_capacity_kwh if p.batt_capacity_kwh > 1e-12 else 0.0
        batt_delta_kwh = e_after - e_before

        # Tank update
        tank_before = state.tank_kwh_th
        tank_loss_kwh = p.tank_loss_per_h * tank_before
        state = thermal_tank_step(state, tank_ch, tank_dis, p)
        tank_after = state.tank_kwh_th
        tank_delta_kwh = tank_after - tank_before

        # -------------------------
        # Electricity accounting (grid closes balance)
        # -------------------------
        total_load_kw = load_kw + hp_p

        # PV first covers load
        pv_to_load = min(pv_kw, total_load_kw)
        remaining_load = total_load_kw - pv_to_load
        pv_surplus = pv_kw - pv_to_load

        # Battery discharge covers remaining load; battery charge consumes PV surplus
        grid_import_kw = max(0.0, remaining_load - batt_dis)
        grid_export_kw = max(0.0, pv_surplus - batt_ch)

        # -------------------------
        # Economics and carbon
        # -------------------------
        import_cost_eur = grid_import_kw * p.dt_h * price
        export_revenue_eur = grid_export_kw * p.dt_h * p.feed_in_tariff_eur_per_kwh
        cost_eur = import_cost_eur - export_revenue_eur

        co2_kg = (grid_import_kw * p.dt_h) * (co2_g / 1000.0)

        # -------------------------
        # Thermal accounting
        # -------------------------
        hp_heat_out_kwh_th = hp_p * cop * p.dt_h
        heat_served_kwh_th = min(heat_d, tank_dis + hp_heat_out_kwh_th)

        rows.append(
            {
                # Exogenous
                "pv_kw": pv_kw,
                "load_base_kw": load_kw,
                "price_import_eur_per_kwh": price,
                "co2_g_per_kwh": co2_g,

                # Dispatch and flows
                "hp_p_kw": hp_p,
                "total_load_kw": total_load_kw,
                "batt_ch_kw": batt_ch,
                "batt_dis_kw": batt_dis,
                "grid_import_kw": grid_import_kw,
                "grid_export_kw": grid_export_kw,

                # Battery state
                "soc": state.soc,
                "batt_delta_kwh": batt_delta_kwh,

                # Tank state
                "tank_kwh_th": state.tank_kwh_th,
                "tank_delta_kwh": tank_delta_kwh,
                "tank_loss_kwh": tank_loss_kwh,
                "tank_charge_kwh_th": tank_ch,
                "tank_dis_kwh_th": tank_dis,

                # Heat KPIs
                "heat_demand_kwh_th": heat_d,
                "heat_served_kwh_th": heat_served_kwh_th,
                "hp_heat_out_kwh_th": hp_heat_out_kwh_th,
                "cop": cop,

                # Techno-economic + carbon
                "import_cost_eur": import_cost_eur,
                "export_revenue_eur": export_revenue_eur,
                "cost_eur": cost_eur,
                "co2_kg": co2_kg,
            }
        )

    out = pd.DataFrame(rows, index=df.index)

    # Cumulative metrics for plotting
    out["cum_cost_eur"] = out["cost_eur"].cumsum()
    out["cum_co2_kg"] = out["co2_kg"].cumsum()

    # Electricity balance residual (should be ~0; used for validation plot)
    out["elec_balance_res_kw"] = (
        out["pv_kw"] + out["batt_dis_kw"] + out["grid_import_kw"]
        - (out["total_load_kw"] + out["batt_ch_kw"] + out["grid_export_kw"])
    )

    return out

In [13]:
# =========================================================
# SENSITIVITY + SCENARIOS (FOR Plotting And Graphs)
# =========================================================

def sensitivity_sweep(
    df: pd.DataFrame,
    base_params: SystemParams,
    batt_sizes: List[float],
    tank_sizes: List[float],
    strategy: Literal["rule", "lp"] = "rule",
) -> pd.DataFrame:
    """
    Simple parameter sweep:
      - battery capacity (kWh)
      - tank capacity (kWh_th)

    Returns a KPI table for plotting diminishing returns / optimal sizing hints.
    """
    records = []
    for b in batt_sizes:
        for t in tank_sizes:
            p = SystemParams(**{**base_params.__dict__, "batt_capacity_kwh": float(b), "tank_capacity_kwh_th": float(t)})

            # If battery capacity is zero, also disable power limits (avoids nonsensical actions)
            if p.batt_capacity_kwh <= 1e-12:
                p = SystemParams(**{**p.__dict__, "batt_p_ch_max_kw": 0.0, "batt_p_dis_max_kw": 0.0})

            res = simulate(df, p, strategy=strategy)
            records.append({
                "batt_capacity_kwh": b,
                "tank_capacity_kwh_th": t,
                "annual_cost_eur": float(res["cost_eur"].sum()),
                "annual_co2_kg": float(res["co2_kg"].sum()),
                "peak_import_kw": float(res["grid_import_kw"].max()),
            })
    return pd.DataFrame(records)


def run_scenarios(
    df: pd.DataFrame,
    p: SystemParams,
    strategy: Literal["rule", "lp"] = "rule",
) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    """
    Scenario analysis modifies exogenous signals to test robustness under:
      - High renewable grid (lower CO2)
      - High carbon winter (higher CO2 in winter)
      - High tariff volatility (higher evening price peaks)
      - Electrification growth (higher base electric load)

    Returns:
      scenarios dict: name -> hourly results df
      summary table: KPI per scenario
    """
    scenarios: Dict[str, pd.DataFrame] = {}
    summaries: List[Dict[str, float]] = []

    def apply_scenario(d: pd.DataFrame, name: str) -> pd.DataFrame:
        dd = d.copy()
        if name == "high_renewable_grid":
            dd["co2_g_per_kwh"] *= 0.7
        elif name == "high_carbon_winter":
            winter = dd.index.month.isin([12, 1, 2])
            dd.loc[winter, "co2_g_per_kwh"] *= 1.3
        elif name == "high_tariff_volatility":
            peak = (dd.index.hour >= 17) & (dd.index.hour <= 21)
            dd.loc[peak, "price_import_eur_per_kwh"] *= 1.25
        elif name == "electrification_growth":
            dd["load_base_kw"] *= 1.15
        else:
            raise ValueError(name)
        return dd

    for name in ["high_renewable_grid", "high_carbon_winter", "high_tariff_volatility", "electrification_growth"]:
        dd = apply_scenario(df, name)
        res = simulate(dd, p, strategy=strategy)
        scenarios[name] = res

        summaries.append({
            "scenario": name,
            "annual_cost_eur": float(res["cost_eur"].sum()),
            "annual_co2_kg": float(res["co2_kg"].sum()),
            "peak_import_kw": float(res["grid_import_kw"].max()),
        })

    return scenarios, pd.DataFrame(summaries)

In [14]:
# =========================================================
# 7) PLOTTING UTILITIES
# =========================================================

def _ensure_dirs() -> None:
    """Create output folders for results and figures."""
    os.makedirs("results", exist_ok=True)
    os.makedirs("results/figures", exist_ok=True)


def _pick_weeks(index: pd.DatetimeIndex) -> Tuple[pd.Timestamp, pd.Timestamp]:
    """
    Choose a representative winter and summer week start.
    Used to standardize week plots across runs.

    Winter: around Jan 15
    Summer: around Jul 15
    """
    year = index[0].year
    winter_start = pd.Timestamp(f"{year}-01-15").floor("D")
    summer_start = pd.Timestamp(f"{year}-07-15").floor("D")

    # If outside range (e.g., partial year), fall back gracefully
    if winter_start < index[0] or winter_start > index[-1]:
        winter_start = index[0].floor("D")
    if summer_start < index[0] or summer_start > index[-1]:
        summer_start = index[len(index)//2].floor("D")

    return winter_start, summer_start

In [15]:
# =========================================================
# 8) PLOTS
# =========================================================

def plot_weekly_powerflows(res: pd.DataFrame, tag: str) -> None:
    """
    Weekly overlay plot of key power flows.

    Why overlay (instead of stacking)?
    - We have bidirectional flows (import/export, charge/discharge),
      and overlay is clearer without extra conventions.
    """
    _ensure_dirs()
    w_start, s_start = _pick_weeks(res.index)

    for start, season in [(w_start, "winter"), (s_start, "summer")]:
        end = start + pd.Timedelta(days=7)
        r = res.loc[start:end]
        if r.empty:
            continue

        fig = plt.figure(figsize=(12, 5))
        ax = plt.gca()

        ax.plot(r.index, r["total_load_kw"], label="Load (kW)")
        ax.plot(r.index, r["pv_kw"], label="PV (kW)")
        ax.plot(r.index, r["grid_import_kw"], label="Grid import (kW)")
        ax.plot(r.index, r["grid_export_kw"], label="Grid export (kW)")
        ax.plot(r.index, r["batt_ch_kw"], label="Battery ch (kW)")
        ax.plot(r.index, r["batt_dis_kw"], label="Battery dis (kW)")
        ax.plot(r.index, r["hp_p_kw"], label="HP power (kW)")

        ax.set_title(f"Weekly power flows ({season}) — {tag}")
        ax.set_xlabel("Time")
        ax.set_ylabel("kW")
        ax.legend(ncols=3, fontsize=9)

        fig.tight_layout()
        fig.savefig(f"results/figures/week_powerflows_{season}_{tag}.png", dpi=200)
        plt.close(fig)


def plot_electric_balance_residual(res: pd.DataFrame, tag: str) -> None:
    """
    Plot electricity balance residual over a week.

    This is the core validation plot:
      residual = pv + batt_dis + grid_import - (load_total + batt_ch + grid_export)

    It should be numerically ~0 (within float precision).
    """
    _ensure_dirs()
    w_start, _ = _pick_weeks(res.index)
    r = res.loc[w_start:w_start + pd.Timedelta(days=7)]
    if r.empty:
        return

    fig = plt.figure(figsize=(12, 3.5))
    ax = plt.gca()
    ax.plot(r.index, r["elec_balance_res_kw"])
    ax.set_title(f"Electricity balance residual (weekly) — {tag}")
    ax.set_xlabel("Time")
    ax.set_ylabel("Residual (kW)")
    fig.tight_layout()
    fig.savefig(f"results/figures/elec_balance_residual_week_{tag}.png", dpi=200)
    plt.close(fig)


def plot_soc_and_tank_week(res: pd.DataFrame, tag: str) -> None:
    """
    Plot battery SOC and tank energy for representative winter/summer weeks.
    Uses twin axes to show SOC (-) and tank energy (kWh_th) simultaneously.
    """
    _ensure_dirs()
    w_start, s_start = _pick_weeks(res.index)

    for start, season in [(w_start, "winter"), (s_start, "summer")]:
        end = start + pd.Timedelta(days=7)
        r = res.loc[start:end]
        if r.empty:
            continue

        fig = plt.figure(figsize=(12, 5))
        ax = plt.gca()
        ax2 = ax.twinx()

        ax.plot(r.index, r["soc"], label="Battery SOC (-)")
        ax2.plot(r.index, r["tank_kwh_th"], label="Tank energy (kWh_th)")

        ax.set_title(f"SOC and tank energy ({season} week) — {tag}")
        ax.set_xlabel("Time")
        ax.set_ylabel("SOC (-)")
        ax2.set_ylabel("Tank (kWh_th)")

        # Combine legends from both axes
        lines, labels = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines + lines2, labels + labels2, ncols=2, fontsize=9, loc="upper left")

        fig.tight_layout()
        fig.savefig(f"results/figures/soc_tank_{season}_week_{tag}.png", dpi=200)
        plt.close(fig)


def plot_soc_histogram(res: pd.DataFrame, tag: str) -> None:
    """Histogram of SOC distribution over the year."""
    _ensure_dirs()
    fig = plt.figure(figsize=(7, 4))
    ax = plt.gca()
    ax.hist(res["soc"].to_numpy(), bins=30)
    ax.set_title(f"Battery SOC distribution — {tag}")
    ax.set_xlabel("SOC (-)")
    ax.set_ylabel("Count")
    fig.tight_layout()
    fig.savefig(f"results/figures/soc_hist_{tag}.png", dpi=200)
    plt.close(fig)


def plot_cumulative_cost_and_co2(rule: pd.DataFrame, opt: Optional[pd.DataFrame]) -> None:
    """
    Cumulative cost and CO2 curves. These are essential for:
      - communicating total annual impact
      - comparing rule vs optimization performance
    """
    _ensure_dirs()

    # Cumulative cost
    fig = plt.figure(figsize=(12, 5))
    ax = plt.gca()
    ax.plot(rule.index, rule["cum_cost_eur"], label="Cumulative cost (rule)")
    if opt is not None:
        ax.plot(opt.index, opt["cum_cost_eur"], label="Cumulative cost (opt)")
    ax.set_title("Cumulative cost over the year")
    ax.set_xlabel("Time")
    ax.set_ylabel("EUR")
    ax.legend()
    fig.tight_layout()
    fig.savefig("results/figures/cum_cost.png", dpi=200)
    plt.close(fig)

    # Cumulative CO2
    fig = plt.figure(figsize=(12, 5))
    ax = plt.gca()
    ax.plot(rule.index, rule["cum_co2_kg"], label="Cumulative CO₂ (rule)")
    if opt is not None:
        ax.plot(opt.index, opt["cum_co2_kg"], label="Cumulative CO₂ (opt)")
    ax.set_title("Cumulative CO₂ emissions over the year")
    ax.set_xlabel("Time")
    ax.set_ylabel("kg CO₂")
    ax.legend()
    fig.tight_layout()
    fig.savefig("results/figures/cum_co2.png", dpi=200)
    plt.close(fig)


def plot_cost_by_hour(res: pd.DataFrame, tag: str) -> None:
    """
    Mean import cost per hour-of-day.
    Shows tariff exposure and demand shifting patterns.
    """
    _ensure_dirs()
    by_h = res.groupby(res.index.hour)["import_cost_eur"].mean()

    fig = plt.figure(figsize=(8, 4))
    ax = plt.gca()
    ax.plot(by_h.index, by_h.values, marker="o")
    ax.set_title(f"Mean import cost per hour-of-day — {tag}")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("EUR per hour (mean)")
    ax.set_xticks(range(0, 24, 2))
    fig.tight_layout()
    fig.savefig(f"results/figures/cost_by_hour_{tag}.png", dpi=200)
    plt.close(fig)


def plot_co2_by_month(res: pd.DataFrame, tag: str) -> None:
    """
    Monthly CO2 totals. Shows seasonal concentration of emissions
    (typically winter-dominant with heat electrification).
    """
    _ensure_dirs()
    by_m = res.resample("MS")["co2_kg"].sum()

    fig = plt.figure(figsize=(10, 4))
    ax = plt.gca()
    ax.plot(by_m.index, by_m.values, marker="o")
    ax.set_title(f"Monthly CO₂ emissions — {tag}")
    ax.set_xlabel("Month")
    ax.set_ylabel("kg CO₂")
    fig.tight_layout()
    fig.savefig(f"results/figures/co2_by_month_{tag}.png", dpi=200)
    plt.close(fig)


def plot_import_duration_curve(baseline: pd.DataFrame, rule: pd.DataFrame, opt: Optional[pd.DataFrame]) -> None:
    """
    Import duration curve = grid import sorted descending.
    Strong visualization for peak shaving and flexibility.
    """
    _ensure_dirs()

    def duration(series: pd.Series) -> np.ndarray:
        return np.sort(series.to_numpy())[::-1]

    d0 = duration(baseline["grid_import_kw"])
    d1 = duration(rule["grid_import_kw"])

    fig = plt.figure(figsize=(10, 5))
    ax = plt.gca()
    ax.plot(d0, label="Baseline (no storage)")
    ax.plot(d1, label="Rule-based")
    if opt is not None:
        ax.plot(duration(opt["grid_import_kw"]), label="Optimized")
    ax.set_title("Grid import duration curve")
    ax.set_xlabel("Hours sorted (descending import)")
    ax.set_ylabel("Grid import (kW)")
    ax.legend()
    fig.tight_layout()
    fig.savefig("results/figures/import_duration_curve.png", dpi=200)
    plt.close(fig)


def plot_import_by_hour(baseline: pd.DataFrame, rule: pd.DataFrame, opt: Optional[pd.DataFrame]) -> None:
    """
    Mean grid import by hour-of-day.
    Highlights load shifting behavior from storage and thermal coupling.
    """
    _ensure_dirs()

    b0 = baseline.groupby(baseline.index.hour)["grid_import_kw"].mean()
    b1 = rule.groupby(rule.index.hour)["grid_import_kw"].mean()

    fig = plt.figure(figsize=(9, 4))
    ax = plt.gca()
    ax.plot(b0.index, b0.values, marker="o", label="Baseline")
    ax.plot(b1.index, b1.values, marker="o", label="Rule")
    if opt is not None:
        b2 = opt.groupby(opt.index.hour)["grid_import_kw"].mean()
        ax.plot(b2.index, b2.values, marker="o", label="Opt")

    ax.set_title("Mean grid import by hour-of-day")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("kW (mean)")
    ax.set_xticks(range(0, 24, 2))
    ax.legend()
    fig.tight_layout()
    fig.savefig("results/figures/import_by_hour.png", dpi=200)
    plt.close(fig)


def plot_dispatch_overlay_week(rule: pd.DataFrame, opt: Optional[pd.DataFrame]) -> None:
    """
    Rule vs optimized dispatch overlay for a winter week.
    Shows differences in:
      - battery charge/discharge timing
      - HP power timing
      - tank state trajectories
    """
    _ensure_dirs()
    if opt is None:
        return

    w_start, _ = _pick_weeks(rule.index)
    end = w_start + pd.Timedelta(days=7)
    r1 = rule.loc[w_start:end]
    r2 = opt.loc[w_start:end]
    if r1.empty or r2.empty:
        return

    # Battery overlay
    fig = plt.figure(figsize=(12, 6))
    ax = plt.gca()
    ax.plot(r1.index, r1["batt_ch_kw"], label="Battery ch (rule)")
    ax.plot(r2.index, r2["batt_ch_kw"], label="Battery ch (opt)")
    ax.plot(r1.index, r1["batt_dis_kw"], label="Battery dis (rule)")
    ax.plot(r2.index, r2["batt_dis_kw"], label="Battery dis (opt)")
    ax.set_title("Dispatch overlay (battery charge/discharge) — winter week")
    ax.set_xlabel("Time")
    ax.set_ylabel("kW")
    ax.legend(ncols=2, fontsize=9)
    fig.tight_layout()
    fig.savefig("results/figures/dispatch_overlay_battery_week.png", dpi=200)
    plt.close(fig)

    # HP + tank overlay
    fig = plt.figure(figsize=(12, 6))
    ax = plt.gca()
    ax2 = ax.twinx()
    ax.plot(r1.index, r1["hp_p_kw"], label="HP power (rule)")
    ax.plot(r2.index, r2["hp_p_kw"], label="HP power (opt)")
    ax2.plot(r1.index, r1["tank_kwh_th"], label="Tank (rule)")
    ax2.plot(r2.index, r2["tank_kwh_th"], label="Tank (opt)")

    ax.set_title("Dispatch overlay (heat pump + tank) — winter week")
    ax.set_xlabel("Time")
    ax.set_ylabel("HP power (kW)")
    ax2.set_ylabel("Tank (kWh_th)")

    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines + lines2, labels + labels2, ncols=2, fontsize=9, loc="upper left")

    fig.tight_layout()
    fig.savefig("results/figures/dispatch_overlay_hp_tank_week.png", dpi=200)
    plt.close(fig)


def plot_sensitivity_curves(sens: pd.DataFrame) -> None:
    """
    Sensitivity plots:
      - battery capacity vs cost / CO2 / peak import
    Plotted separately for each tank capacity.
    """
    _ensure_dirs()

    for tank in sorted(sens["tank_capacity_kwh_th"].unique()):
        d = sens[sens["tank_capacity_kwh_th"] == tank].sort_values("batt_capacity_kwh")

        fig = plt.figure(figsize=(9, 4))
        ax = plt.gca()
        ax.plot(d["batt_capacity_kwh"], d["annual_cost_eur"], marker="o")
        ax.set_title(f"Sensitivity: Battery size vs annual cost (tank={tank} kWh_th)")
        ax.set_xlabel("Battery capacity (kWh)")
        ax.set_ylabel("Annual cost (EUR)")
        fig.tight_layout()
        fig.savefig(f"results/figures/sens_batt_vs_cost_tank{int(tank)}.png", dpi=200)
        plt.close(fig)

        fig = plt.figure(figsize=(9, 4))
        ax = plt.gca()
        ax.plot(d["batt_capacity_kwh"], d["annual_co2_kg"], marker="o")
        ax.set_title(f"Sensitivity: Battery size vs annual CO₂ (tank={tank} kWh_th)")
        ax.set_xlabel("Battery capacity (kWh)")
        ax.set_ylabel("Annual CO₂ (kg)")
        fig.tight_layout()
        fig.savefig(f"results/figures/sens_batt_vs_co2_tank{int(tank)}.png", dpi=200)
        plt.close(fig)

        fig = plt.figure(figsize=(9, 4))
        ax = plt.gca()
        ax.plot(d["batt_capacity_kwh"], d["peak_import_kw"], marker="o")
        ax.set_title(f"Sensitivity: Battery size vs peak import (tank={tank} kWh_th)")
        ax.set_xlabel("Battery capacity (kWh)")
        ax.set_ylabel("Peak import (kW)")
        fig.tight_layout()
        fig.savefig(f"results/figures/sens_batt_vs_peak_tank{int(tank)}.png", dpi=200)
        plt.close(fig)


def plot_scenario_comparison(summaries: pd.DataFrame) -> None:
    """
    Scenario comparison bar charts:
      - annual cost
      - annual CO2
      - peak import
    """
    _ensure_dirs()
    if summaries.empty:
        return

    # Annual cost
    fig = plt.figure(figsize=(10, 4))
    ax = plt.gca()
    ax.bar(summaries["scenario"], summaries["annual_cost_eur"])
    ax.set_title("Scenario comparison: annual cost")
    ax.set_ylabel("EUR")
    ax.set_xticklabels(summaries["scenario"], rotation=20, ha="right")
    fig.tight_layout()
    fig.savefig("results/figures/scenario_cost.png", dpi=200)
    plt.close(fig)

    # Annual CO2
    fig = plt.figure(figsize=(10, 4))
    ax = plt.gca()
    ax.bar(summaries["scenario"], summaries["annual_co2_kg"])
    ax.set_title("Scenario comparison: annual CO₂")
    ax.set_ylabel("kg CO₂")
    ax.set_xticklabels(summaries["scenario"], rotation=20, ha="right")
    fig.tight_layout()
    fig.savefig("results/figures/scenario_co2.png", dpi=200)
    plt.close(fig)

    # Peak import
    fig = plt.figure(figsize=(10, 4))
    ax = plt.gca()
    ax.bar(summaries["scenario"], summaries["peak_import_kw"])
    ax.set_title("Scenario comparison: peak grid import")
    ax.set_ylabel("kW")
    ax.set_xticklabels(summaries["scenario"], rotation=20, ha="right")
    fig.tight_layout()
    fig.savefig("results/figures/scenario_peak.png", dpi=200)
    plt.close(fig)


In [16]:
# =========================================================
# MAIN RUNNER
# =========================================================

def run_full_project() -> None:
    """
    Full pipeline:
      1) Generate synthetic year profiles
      2) Run baseline (no storage)
      3) Run rule-based strategy
      4) Run optimization strategy (if SciPy available)
      5) Save CSV outputs
      6) Create all requested plots
      7) Run sensitivity and scenario analyses + plots
    """
    _ensure_dirs()

    # Base parameters (edit for different system sizing)
    p = SystemParams()

    # Generate exogenous signals for the year
    df = generate_synthetic_berlin_profiles(year=2025, seed=2)

    # Baseline (no storage / no flexibility)
    baseline = simulate(
        df,
        SystemParams(**{**p.__dict__, "batt_capacity_kwh": 0.0, "batt_p_ch_max_kw": 0.0, "batt_p_dis_max_kw": 0.0,
                        "tank_capacity_kwh_th": p.tank_min_kwh_th}),
        strategy="rule",
        thermal_precharge_rule=False,
    )

    # Rule-based operation
    rule = simulate(df, p, strategy="rule", thermal_precharge_rule=True)

    # Optimization-based operation (optional)
    opt = None
    if SCIPY_AVAILABLE:
        opt = simulate(df, p, strategy="lp", co2_penalty_eur_per_kg=0.0)

    # -------------------------
    # Save time-series outputs
    # -------------------------
    baseline.to_csv("results/results_baseline_hourly.csv")
    rule.to_csv("results/results_rule_hourly.csv")
    if opt is not None:
        opt.to_csv("results/results_opt_hourly.csv")

    # -------------------------
    # Plots: validation + operations + economics + carbon + flexibility + benchmarking
    # -------------------------
    plot_weekly_powerflows(rule, tag="rule")
    plot_electric_balance_residual(rule, tag="rule")
    plot_soc_and_tank_week(rule, tag="rule")
    plot_soc_histogram(rule, tag="rule")
    plot_cost_by_hour(rule, tag="rule")
    plot_co2_by_month(rule, tag="rule")

    if opt is not None:
        plot_weekly_powerflows(opt, tag="opt")
        plot_electric_balance_residual(opt, tag="opt")
        plot_soc_and_tank_week(opt, tag="opt")
        plot_soc_histogram(opt, tag="opt")
        plot_cost_by_hour(opt, tag="opt")
        plot_co2_by_month(opt, tag="opt")

    plot_cumulative_cost_and_co2(rule, opt)
    plot_import_duration_curve(baseline, rule, opt)
    plot_import_by_hour(baseline, rule, opt)
    plot_dispatch_overlay_week(rule, opt)

    # -------------------------
    # Sensitivity analysis (use 60-day subset for speed; switch to full df for full-year sweeps)
    # -------------------------
    df_short = df.iloc[:24 * 60]
    sens = sensitivity_sweep(
        df_short,
        base_params=p,
        batt_sizes=[0, 5, 10, 15],
        tank_sizes=[15, 30],
        strategy="rule",
    )
    sens.to_csv("results/sensitivity_table.csv", index=False)
    plot_sensitivity_curves(sens)

    # -------------------------
    # Scenario analysis (full year)
    # -------------------------
    _, scen_summary = run_scenarios(df, p, strategy="rule")
    scen_summary.to_csv("results/scenario_summary.csv", index=False)
    plot_scenario_comparison(scen_summary)

    # Final note for the user
    print("Done.")
    print("Figures saved to: results/figures/")
    print("CSVs saved to: results/")


if __name__ == "__main__":
    run_full_project()


/tmp/ipython-input-3013375040.py:367: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(summaries["scenario"], rotation=20, ha="right")
/tmp/ipython-input-3013375040.py:378: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(summaries["scenario"], rotation=20, ha="right")


Done.
Figures saved to: results/figures/
CSVs saved to: results/


/tmp/ipython-input-3013375040.py:389: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(summaries["scenario"], rotation=20, ha="right")
